# DQE state optimisation with an adjoint entropy gradient

This notebook checks the four-spin adjoint against the simulator and central differences, then runs the entropy-constrained DQE optimiser. The cost is

$$C(x)=\frac{1}{N+1}\sum_{k=0}^{N}\left[\sum_{P\in\mathcal P}S_P(\psi_k)-S_{\max}\right]_+^2,$$

where $\mathcal P$ contains the seven distinct bipartitions of four qubits. The adjoint differentiates every reduced-state entropy and the simulator's ordered drift–exchange Trotter step. The derivation is in `DQE_STATE_ADJOINT.md`.


In [ ]:
import sys
import time
from pathlib import Path
import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pulse_vqe_entanglement").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pulse_vqe_entanglement.dataclasses import SimulationConfig
from pulse_vqe_entanglement.dqe_state import (
    _make_system_and_observable, _make_driven_sys, _cost_entropy, _entropy_grad_fd,
)
from pulse_vqe_entanglement.dqe_state_adjoint import dqe_optimisation
from pulse_vqe_entanglement.dqe_state_entropy_gradient import entropy_cost_and_grad
from pulse_vqe_entanglement.entanglement_measures import (
    ALL_BIPARTITIONS_4Q, entropy_bipartite_traj, entropy_sum_traj,
)
from pulse_vqe_entanglement.plotting_state import plot_dqe_optimisation


## Check the gradient and trajectory

Both tests use a nonzero entropy hinge cost. The second test reverses the exchange sign and permits negative couplings. Central differences propagate the simulator twice for each of the nine controls.


In [ ]:
initial = np.eye(16, dtype=np.complex128)[:, 3]  # |0011⟩
rng = np.random.default_rng(1234)
for feromagnetic, j_min in ((False, 0.0), (True, -4.0)):
    check_config = SimulationConfig(
        zeeman_splittings=np.array([1000.0, 1008.0, 1016.0, 1024.0]),
        initial_state=initial,
        n_time_steps=3, n_discretisation=2, n_J_channels=3,
        T=70.0, J_min=j_min, J_max=10.0,
        feromagnetic=feromagnetic, S_max=0.05,
    )
    check_x = rng.uniform(-0.5, 0.5, 9)
    system, observable = _make_system_and_observable(check_config)
    driven = _make_driven_sys(check_config, system)
    driven.propagate_all(check_x)  # compile the simulator before timing

    start = time.perf_counter()
    cost, analytic_grad, model_traj = entropy_cost_and_grad(
        check_config, check_x, return_trajectory=True
    )
    adjoint_seconds = time.perf_counter() - start
    sim_traj = np.asarray(driven.propagate_all(check_x))
    sim_cost = _cost_entropy(driven, check_x, S_max=check_config.S_max)
    start = time.perf_counter()
    fd_grad = _entropy_grad_fd(driven, check_x, S_max=check_config.S_max, eps=1e-5)
    fd_seconds = time.perf_counter() - start

    trajectory_error = np.max(np.abs(model_traj - sim_traj))
    gradient_error = np.max(np.abs(analytic_grad - fd_grad))
    print(f"feromagnetic={feromagnetic}, J_min={j_min:g}: "
          f"trajectory error={trajectory_error:.2e}, "
          f"cost error={abs(cost - sim_cost):.2e}, "
          f"gradient error={gradient_error:.2e}, "
          f"adjoint={adjoint_seconds:.3f}s, FD={fd_seconds:.3f}s")
    assert trajectory_error < 1e-7
    assert abs(cost - sim_cost) < 1e-8
    assert np.allclose(analytic_grad, fd_grad, atol=1e-5, rtol=1e-4)


## Run the augmented Lagrangian optimiser

The fidelity-only baseline and entropy stage start from the same initial pulse. This example uses the positive-exchange DQE setup from the original DQE notebook. For the target $\cos\theta|0011\rangle+\sin\theta|1100\rangle$, every one of the seven cuts has terminal entropy $h_2(\sin^2\theta)$, so the sum has a floor of $7h_2(\sin^2\theta)\approx0.0789$ bits at $\theta=0.01\pi$. The cap is set just above that floor. The larger inner-iteration allowance matters: 100 iterations stopped while the entropy cost was still falling.


In [ ]:
config = SimulationConfig(
    zeeman_splittings=np.array([1000.0, 1008.0, 1016.0, 1024.0]),
    initial_state=initial,
    theta_target=0.01 * np.pi,
    n_time_steps=50, n_discretisation=1, T=500.0,
    max_drive_strength=0.0,
    J_min=0.0, J_max=10.0,
    J_init_scale=0.1, J_init_mean=0.0, J_init_seed=1234,
    feromagnetic=False,
    gtol=1e-12, maxiter=500,
    entanglement_optimisation=True,
    S_max=0.08, eps_infidelity=1e-8,
    rho=0.0, dual_lr=1e10,
    n_outer=3, inner_maxiter=300,
    dqe_plot_partitions=[(0, 1), (0,)],
)
start = time.perf_counter()
results = dqe_optimisation(config)
elapsed = time.perf_counter() - start
baseline_entropy = entropy_sum_traj(results.baseline.final_traj)
final_entropy = entropy_sum_traj(results.final_traj)
print(f"Baseline infidelity: {float(results.baseline.val_final[-1]):.3e}")
print(f"Final infidelity:    {float(results.val_final[-1]):.3e}")
print(f"Baseline cost:       {np.mean(np.maximum(baseline_entropy - config.S_max, 0)**2):.3e}")
print(f"Final cost:          {results.history[-1]['S']:.3e}")
target_probability = np.sin(config.theta_target)**2
target_cut_entropy = -(target_probability * np.log2(target_probability)
                       + (1 - target_probability) * np.log2(1 - target_probability))
print(f"Target entropy floor: {7 * target_cut_entropy:.3e} bits (sum of seven cuts)")
print(f"Fidelity feasible:   {results.optimization_success}")
print(f"Inner solves converged: {[row['inner_success'] for row in results.history]}")
print("Cut       Baseline mean   Adjoint mean   Baseline peak   Adjoint peak")
for cut in ALL_BIPARTITIONS_4Q:
    before = entropy_bipartite_traj(results.baseline.final_traj, subsystem=cut)
    after = entropy_bipartite_traj(results.final_traj, subsystem=cut)
    print(f"{str(cut):<9} {before.mean():13.4f} {after.mean():14.4f} "
          f"{before.max():15.4f} {after.max():14.4f}")
print(f"Elapsed:             {elapsed:.2f} s")
plot_dqe_optimisation(results)
